# Pratica 07 — Analisi grammaticale ed entità

**Corso:** NLP Classico con Python · **Lezione 07** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase07_analisis_gramatical_entidades_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi da 1 a 5 non richiedono modelli: usano cifre, insiemi e alberi scritti a mano e `spacy.blank`. La sfida usa il corpus di `datos/` (in Colab, carica la cartella `datos/`). I testi sono sintetici e scritti per il corso (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas` e `spacy`.

---
## Cella 0 — Setup e dati

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("spacy", "spacy")
print("Librerie pronte.")

In [ ]:
import os
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
import spacy
from spacy.tokens import Doc
from spacy.matcher import Matcher

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

MARCA = re.compile(r"\{([^{}|]+)\|(PER|ORG|LOC|MISC)\}")

def parsear(marcado):
    """Converte 'Ciao {Ana|PER}' in (testo, {(inizio, fine, etichetta)})."""
    texto, ents, pos = "", set(), 0
    for m in MARCA.finditer(marcado):
        texto += marcado[pos:m.start()]
        ents.add((len(texto), len(texto) + len(m.group(1)), m.group(2)))
        texto += m.group(1)
        pos = m.end()
    return texto + marcado[pos:], ents

ETIQUETAS = ["PER", "ORG", "LOC", "MISC"]
print("Librerie caricate correttamente.")

In [ ]:
# datos-es
# Quattro frasi di esempio: riferimento (gold) e previsione di un modello immaginario
EJEMPLO = [
    ("{Marta Vidal|PER} dirige la {Fundación Horizonte|ORG} en {Quito|LOC}.",
     "{Marta Vidal|PER} dirige la {Fundación|ORG} Horizonte en {Quito|LOC}."),
    ("Los {catalanes|MISC} visitaron {Lisboa|LOC}.",
     "Los catalanes visitaron {Lisboa|LOC}."),
    ("{Pedro Soler|PER} viajó a {Toledo|LOC}.",
     "{Pedro Soler|PER} viajó a {Toledo|ORG}."),
    ("Hoy hace buen tiempo.",
     "{Hoy|PER} hace buen tiempo."),
]
GOLD_EJ, PRED_EJ = [], []
for _ref, _pred in EJEMPLO:
    _t1, _g = parsear(_ref)
    _t2, _p = parsear(_pred)
    assert _t1 == _t2
    GOLD_EJ.append(_g); PRED_EJ.append(_p)

# Due alberi scritti a mano (teste assolute): spagnolo e italiano
def _arbol(idioma, palabras, espacios, heads, deps, pos, lemmas):
    return Doc(spacy.blank(idioma).vocab, words=palabras, spaces=espacios, heads=heads, deps=deps, pos=pos, lemmas=lemmas)

DOC_ES = _arbol("es", ["La", "alcaldesa", "inauguró", "la", "feria", "y", "el", "concejal", "visitó", "el", "museo", "."],
                [True] * 11 + [False],
                [1, 2, 2, 4, 2, 8, 7, 8, 2, 10, 8, 2],
                ["det", "nsubj", "ROOT", "det", "obj", "cc", "det", "nsubj", "conj", "det", "obj", "punct"],
                ["DET", "NOUN", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "VERB", "DET", "NOUN", "PUNCT"],
                ["el", "alcaldesa", "inaugurar", "el", "feria", "y", "el", "concejal", "visitar", "el", "museo", "."])
DOC_IT = _arbol("it", ["Il", "sindaco", "ha", "inaugurato", "la", "fiera", "e", "l'", "assessore", "ha", "visitato", "il", "museo", "."],
                [True, True, True, True, True, True, True, False, True, True, True, True, False, False],
                [1, 3, 3, 3, 5, 3, 10, 8, 10, 10, 3, 12, 10, 3],
                ["det", "nsubj", "aux", "ROOT", "det", "obj", "cc", "det", "nsubj", "aux", "conj", "det", "obj", "punct"],
                ["DET", "NOUN", "AUX", "VERB", "DET", "NOUN", "CCONJ", "DET", "NOUN", "AUX", "VERB", "DET", "NOUN", "PUNCT"],
                ["il", "sindaco", "avere", "inaugurare", "il", "fiera", "e", "il", "assessore", "avere", "visitare", "il", "museo", "."])

## Esercizio 1 — Precisione, richiamo e F1

Completa `prf(tp, fp, fn)`: deve restituire la tupla `(precisione, richiamo, F1)`. Se un denominatore è 0, il valore corrispondente è `0.0` (e la F1 è `0.0` se precisione + richiamo è 0).

Prima di programmare, calcola a mano con `tp = 4`, `fp = 3`, `fn = 3` (le cifre della lezione): P, R e F1 valgono 4/7.

In [ ]:
def prf(tp, fp, fn):
    precision = None      # TODO: tp / (tp + fp), o 0.0 se il denominatore è 0
    recall = None         # TODO: tp / (tp + fn), o 0.0 se il denominatore è 0
    f1 = None             # TODO: media armonica delle due
    return precision, recall, f1

# Controllo automatico
_r = prf(4, 3, 3)
assert None not in _r, "Completa le righe contrassegnate con TODO"
assert np.allclose(_r, (4 / 7, 4 / 7, 4 / 7)), "con tp=4, fp=3, fn=3 i tre valori sono 4/7"
assert np.allclose(prf(3, 1, 0), (0.75, 1.0, 6 / 7)), "con tp=3, fp=1, fn=0: P=0,75, R=1 e F1=6/7"
assert prf(0, 0, 1) == (0.0, 0.0, 0.0) and prf(0, 0, 0) == (0.0, 0.0, 0.0), "attenzione alla divisione per zero"
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Contare TP, FP e FN per etichetta

Completa `contar_exacto(gold, pred)`. `gold` e `pred` sono liste (una posizione per frase) di insiemi di entità `(inizio, fine, etichetta)`. Restituisce un dizionario `{etichetta: (tp, fp, fn)}` con le quattro etichette di `ETIQUETAS`, in modo **esatto** (span ed etichetta uguali).

Suggerimento: per frase, i TP sono `g & p`, i FP sono `p - g` e i FN sono `g - p`. Si verifica con `GOLD_EJ` e `PRED_EJ`: conta prima a mano il totale (4 TP, 3 FP, 3 FN).

In [ ]:
def contar_exacto(gold, pred):
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        pass    # TODO: somma in cuenta[etichetta][0], [1] o [2] gli elementi di g & p, p - g e g - p
    return None      # TODO: restituisci {etichetta: tuple(conteggio)}

cuenta_ej = contar_exacto(GOLD_EJ, PRED_EJ)

# Controllo automatico
assert cuenta_ej is not None, "Completa le righe contrassegnate con TODO"
assert cuenta_ej == {"PER": (2, 1, 0), "ORG": (0, 2, 1), "LOC": (2, 0, 1), "MISC": (0, 0, 1)}, "rivedi il conteggio per etichetta (un errore di etichetta è FP di un'etichetta e FN di un'altra)"
assert tuple(sum(v[k] for v in cuenta_ej.values()) for k in range(3)) == (4, 3, 3), "il totale deve essere 4 TP, 3 FP e 3 FN"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Tipi di errore

Completa `tipo_error(g, predichas)`. `g` è un'entità del riferimento `(inizio, fine, etichetta)` che **non** è stata azzeccata, e `predichas` è l'insieme delle entità previste in quella frase. Restituisce:

- `"etiqueta"` se qualche prevista ha lo **stesso span** (con un'altra etichetta),
- `"frontera"` se qualche prevista si **sovrappone** a `g` ma lo span non coincide,
- `"omitida"` se nessuna prevista si sovrappone a `g`.

Due span `(a1, b1)` e `(a2, b2)` si sovrappongono se `a1 < b2 and a2 < b1`.

In [ ]:
def tipo_error(g, predichas):
    # TODO: controlla prima lo stesso span, poi la sovrapposizione
    return None

# Controllo automatico
_tipos = [tipo_error(g, p) for gs, p in zip(GOLD_EJ, PRED_EJ) for g in gs - p]
assert None not in _tipos, "Completa le righe contrassegnate con TODO"
assert sorted(_tipos) == ["etiqueta", "frontera", "omitida"], "la prima entità è di confine, la seconda omessa e la terza di etichetta"
assert tipo_error((0, 5, "PER"), {(0, 5, "LOC")}) == "etiqueta" and tipo_error((0, 5, "PER"), {(3, 9, "PER")}) == "frontera" and tipo_error((0, 5, "PER"), {(5, 9, "PER")}) == "omitida", "gli span (0, 5) e (5, 9) non si sovrappongono"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Soggetto, verbo e oggetto dall'albero

`DOC_ES` e `DOC_IT` sono alberi scritti a mano (non serve nessun modello). Completa `extraer_svo(doc)`: per ogni token con `pos_ == "VERB"`, cerca tra i suoi figli (`token.children`) uno con `dep_` che inizia per `nsubj` e uno con `dep_ == "obj"`; se ci sono entrambi, aggiungi la terna `(testo del sottoalbero del soggetto, lemma del verbo, testo del sottoalbero dell'oggetto)`.

Per ottenere il testo di un sottoalbero usa `"".join(x.text_with_ws for x in token.subtree).strip()`. L'ausiliare italiano (`ha`) non è `VERB`: non considerarlo.

In [ ]:
def extraer_svo(doc):
    ternas = []
    for t in doc:
        pass    # TODO: se t è VERB e ha un figlio nsubj e un figlio obj, aggiungi (soggetto, t.lemma_, oggetto)
    return None      # TODO: restituisci la lista di terne

svo_es = extraer_svo(DOC_ES)
svo_it = extraer_svo(DOC_IT)

# Controllo automatico
assert svo_es is not None and svo_it is not None, "Completa le righe contrassegnate con TODO"
assert svo_es == [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")], "l'albero spagnolo ha due terne"   # dato-es
assert svo_it == [("Il sindaco", "inaugurare", "la fiera"), ("l'assessore", "visitare", "il museo")], "l'albero italiano ha due terne; l'ausiliare non conta"   # dato-es
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Un `EntityRuler` e un `Matcher`

Senza modelli: usa `spacy.blank(idioma)`, che si limita a tokenizzare.

- Completa `crear_nlp_reglas(idioma, siglas, gentilicios)`: restituisce una *pipeline* vuota con un `entity_ruler` che etichetta ogni sigla come `ORG` (pattern `[{"ORTH": sigla}]`) e ogni gentile come `MISC` (pattern `[{"LOWER": {"IN": gentilicios}}]`).
- Completa `buscar_titulos(nlp, texto)`: con un `Matcher`, restituisce i testi delle sequenze di una o più parole con l'iniziale maiuscola (`{"IS_TITLE": True, "OP": "+"}`) che **seguono** la parola `de`, tenendo le sequenze più lunghe (`spacy.util.filter_spans`).

In [ ]:
def crear_nlp_reglas(idioma, siglas, gentilicios):
    nlp = spacy.blank(idioma)
    ruler = nlp.add_pipe("entity_ruler")
    patrones = None      # TODO: lista di {"label": ..., "pattern": [...]}: una per sigla e una per i gentili
    ruler.add_patterns(patrones)
    return nlp

def buscar_titulos(nlp, texto):
    matcher = Matcher(nlp.vocab)
    # TODO: aggiungi il pattern [{"LOWER": "de"}, {"IS_TITLE": True, "OP": "+"}] e filtra con filter_spans; ricorda di togliere la parola "de"
    return None

nlp_es = crear_nlp_reglas("es", ["ONU", "UE"], ["italianos", "catalanes"])      # dato-es
ents_es = [(e.text, e.label_) for e in nlp_es("La ONU y los italianos hablaron con la UE.").ents]      # dato-es
titulos = buscar_titulos(spacy.blank("es"), "La Universidad de Salamanca y el Museo de Historia Natural.")      # dato-es

# Controllo automatico
assert titulos is not None, "Completa le righe contrassegnate con TODO"
assert ents_es == [("ONU", "ORG"), ("italianos", "MISC"), ("UE", "ORG")], "il ruler deve etichettare ONU e UE come ORG e italianos come MISC"
assert titulos == ["Salamanca", "Historia Natural"], "il Matcher deve restituire le sequenze con maiuscola dopo «de», senza la parola «de»"   # dato-es
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Valutare un sistema di regole sul corpus reale

Con `datos/noticias_ner_es.csv` e `datos/noticias_ner_it.csv` (riferimento annotato a mano):

1. Scrivi `evaluar_corpus(idioma, nlp)`: legge il corpus di quella lingua (`"es"` o `"it"`), elabora ogni testo con `nlp` e restituisce `(precisione, richiamo, f1)` micro in modo esatto, usando le tue funzioni `contar_exacto` e `prf` degli esercizi 1 e 2. Restituisce anche il numero di entità del riferimento.
2. Valuta una *pipeline* **solo di regole** (`crear_nlp_reglas` con le sigle `ONU`, `OMS`, `FMI`, `UE`, `UNESCO`, `UGT` e `CGIL` e senza gentili) in ciascuna lingua.

Pensa cosa ti aspetti: senza modello statistico, quale dovrebbe essere la precisione? E il richiamo? Si verifica con un calcolo indipendente sul riferimento.

> **Nota:** il corpus è piccolo e sintetico (*esempio illustrativo*); le cifre non si generalizzano.

In [ ]:
SIGLAS_RETO = ["ONU", "OMS", "FMI", "UE", "UNESCO", "UGT", "CGIL"]

def evaluar_corpus(idioma, nlp):
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    gold = [{(a, b, e) for a, b, e in json.loads(x)} for x in df["entidades"]]
    # TODO: prevedi con nlp.pipe(df["texto"]) e calcola (P, R, F1) con contar_exacto e prf
    return None, sum(len(g) for g in gold)

resultado = {i: evaluar_corpus(i, crear_nlp_reglas(i, SIGLAS_RETO, [])) for i in ("es", "it")}      # dato-es

# Controllo automatico: calcolo indipendente sul riferimento
assert all(v[0] is not None for v in resultado.values()), "Completa le righe contrassegnate con TODO"
for idioma, ((p, r, f), n_gold) in resultado.items():
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    aciertos = sum(1 for t, x in zip(df["texto"], df["entidades"]) for a, b, e in json.loads(x) if e == "ORG" and t[a:b] in SIGLAS_RETO)
    assert aciertos > 5 and n_gold >= 100
    assert np.isclose(p, 1.0), "tutte le sigle della lista sono ORG nel riferimento: precisione 1"
    assert np.isclose(r, aciertos / n_gold), "il richiamo è successi / entità del riferimento"
    assert np.isclose(f, 2 * p * r / (p + r))
print("Sfida finale corretta ✔")
print({i: tuple(round(x, 3) for x in v[0]) for i, v in resultado.items()})

**Le tue conclusioni (scrivi 3-4 frasi):** una *pipeline* solo di regole ha precisione perfetta e richiamo molto basso. Perché? In cosa si differenzia da ciò che accade con un modello statistico? Come combineresti regole e modello e con quali frasi misureresti se la combinazione migliora?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

# Controllo automatico
_r = prf(4, 3, 3)
assert None not in _r, "Completa le righe contrassegnate con TODO"
assert np.allclose(_r, (4 / 7, 4 / 7, 4 / 7)), "con tp=4, fp=3, fn=3 i tre valori sono 4/7"
assert np.allclose(prf(3, 1, 0), (0.75, 1.0, 6 / 7)), "con tp=3, fp=1, fn=0: P=0,75, R=1 e F1=6/7"
assert prf(0, 0, 1) == (0.0, 0.0, 0.0) and prf(0, 0, 0) == (0.0, 0.0, 0.0), "attenzione alla divisione per zero"
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def contar_exacto(gold, pred):
    cuenta = {e: [0, 0, 0] for e in ETIQUETAS}
    for g, p in zip(gold, pred):
        for _, _, e in g & p:
            cuenta[e][0] += 1
        for _, _, e in p - g:
            cuenta[e][1] += 1
        for _, _, e in g - p:
            cuenta[e][2] += 1
    return {e: tuple(v) for e, v in cuenta.items()}

cuenta_ej = contar_exacto(GOLD_EJ, PRED_EJ)

# Controllo automatico
assert cuenta_ej is not None, "Completa le righe contrassegnate con TODO"
assert cuenta_ej == {"PER": (2, 1, 0), "ORG": (0, 2, 1), "LOC": (2, 0, 1), "MISC": (0, 0, 1)}, "rivedi il conteggio per etichetta (un errore di etichetta è FP di un'etichetta e FN di un'altra)"
assert tuple(sum(v[k] for v in cuenta_ej.values()) for k in range(3)) == (4, 3, 3), "il totale deve essere 4 TP, 3 FP e 3 FN"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def tipo_error(g, predichas):
    if any(g[:2] == p[:2] for p in predichas):
        return "etiqueta"
    if any(g[0] < p[1] and p[0] < g[1] for p in predichas):
        return "frontera"
    return "omitida"

# Controllo automatico
_tipos = [tipo_error(g, p) for gs, p in zip(GOLD_EJ, PRED_EJ) for g in gs - p]
assert None not in _tipos, "Completa le righe contrassegnate con TODO"
assert sorted(_tipos) == ["etiqueta", "frontera", "omitida"], "la prima entità è di confine, la seconda omessa e la terza di etichetta"
assert tipo_error((0, 5, "PER"), {(0, 5, "LOC")}) == "etiqueta" and tipo_error((0, 5, "PER"), {(3, 9, "PER")}) == "frontera" and tipo_error((0, 5, "PER"), {(5, 9, "PER")}) == "omitida", "gli span (0, 5) e (5, 9) non si sovrappongono"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def extraer_svo(doc):
    ternas = []
    for t in doc:
        if t.pos_ != "VERB":
            continue
        sujetos = [h for h in t.children if h.dep_.startswith("nsubj")]
        objetos = [h for h in t.children if h.dep_ == "obj"]
        if sujetos and objetos:
            sujeto = "".join(x.text_with_ws for x in sujetos[0].subtree).strip()
            objeto = "".join(x.text_with_ws for x in objetos[0].subtree).strip()
            ternas.append((sujeto, t.lemma_, objeto))
    return ternas

svo_es = extraer_svo(DOC_ES)
svo_it = extraer_svo(DOC_IT)

# Controllo automatico
assert svo_es is not None and svo_it is not None, "Completa le righe contrassegnate con TODO"
assert svo_es == [("La alcaldesa", "inaugurar", "la feria"), ("el concejal", "visitar", "el museo")], "l'albero spagnolo ha due terne"   # dato-es
assert svo_it == [("Il sindaco", "inaugurare", "la fiera"), ("l'assessore", "visitare", "il museo")], "l'albero italiano ha due terne; l'ausiliare non conta"   # dato-es
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def crear_nlp_reglas(idioma, siglas, gentilicios):
    nlp = spacy.blank(idioma)
    ruler = nlp.add_pipe("entity_ruler")
    patrones = [{"label": "ORG", "pattern": [{"ORTH": s}]} for s in siglas]
    patrones.append({"label": "MISC", "pattern": [{"LOWER": {"IN": gentilicios}}]})
    ruler.add_patterns(patrones)
    return nlp

def buscar_titulos(nlp, texto):
    from spacy.util import filter_spans
    matcher = Matcher(nlp.vocab)
    matcher.add("DE_TITULO", [[{"LOWER": "de"}, {"IS_TITLE": True, "OP": "+"}]])
    doc = nlp(texto)
    tramos = filter_spans([doc[a:b] for _, a, b in matcher(doc)])
    return [t[1:].text for t in tramos]

nlp_es = crear_nlp_reglas("es", ["ONU", "UE"], ["italianos", "catalanes"])      # dato-es
ents_es = [(e.text, e.label_) for e in nlp_es("La ONU y los italianos hablaron con la UE.").ents]      # dato-es
titulos = buscar_titulos(spacy.blank("es"), "La Universidad de Salamanca y el Museo de Historia Natural.")      # dato-es

# Controllo automatico
assert titulos is not None, "Completa le righe contrassegnate con TODO"
assert ents_es == [("ONU", "ORG"), ("italianos", "MISC"), ("UE", "ORG")], "il ruler deve etichettare ONU e UE come ORG e italianos come MISC"
assert titulos == ["Salamanca", "Historia Natural"], "il Matcher deve restituire le sequenze con maiuscola dopo «de», senza la parola «de»"   # dato-es
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
SIGLAS_RETO = ["ONU", "OMS", "FMI", "UE", "UNESCO", "UGT", "CGIL"]

def evaluar_corpus(idioma, nlp):
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    gold = [{(a, b, e) for a, b, e in json.loads(x)} for x in df["entidades"]]
    pred = [{(e.start_char, e.end_char, e.label_) for e in doc.ents} for doc in nlp.pipe(df["texto"])]
    cuenta = contar_exacto(gold, pred)
    tp, fp, fn = (sum(v[k] for v in cuenta.values()) for k in range(3))
    return prf(tp, fp, fn), sum(len(g) for g in gold)

resultado = {i: evaluar_corpus(i, crear_nlp_reglas(i, SIGLAS_RETO, [])) for i in ("es", "it")}      # dato-es

assert all(v[0] is not None for v in resultado.values()), "Completa le righe contrassegnate con TODO"
for idioma, ((p, r, f), n_gold) in resultado.items():
    df = pd.read_csv(cargar_datos(f"noticias_ner_{idioma}.csv"))
    aciertos = sum(1 for t, x in zip(df["texto"], df["entidades"]) for a, b, e in json.loads(x) if e == "ORG" and t[a:b] in SIGLAS_RETO)
    assert aciertos > 5 and n_gold >= 100
    assert np.isclose(p, 1.0), "tutte le sigle della lista sono ORG nel riferimento: precisione 1"
    assert np.isclose(r, aciertos / n_gold), "il richiamo è successi / entità del riferimento"
    assert np.isclose(f, 2 * p * r / (p + r))
print("Sfida finale corretta ✔")
print({i: tuple(round(x, 3) for x in v[0]) for i, v in resultado.items()})

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (POS contro dipendenze, successo esatto contro parziale, precisione contro richiamo, etichetta contro confine, `Matcher` contro `EntityRuler`)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*